In [ ]:
import blackjax
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from jax import vmap
from src.fdm import EMechanismFDMSolver
from src.params import EMechanismFDMParams
from src.utils import generate_noisy_samples
from src.voltammetry import CyclicDC, LinearSweepAC, LinearSweepDC

plt.style.use("seaborn-v0_8-colorblind")

# Effect of each parameter


In [ ]:
voltammetry = LinearSweepDC()

fdm_solver = EMechanismFDMSolver(voltammetry)

base_params = EMechanismFDMParams(
    alpha=jnp.array(0.6), K0=jnp.array(1.0), E0=jnp.array(2.0), dB=jnp.array(0.5)
)

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))

# Alpha Varying

alpha_range = jnp.linspace(0.3, 0.7, 5)
alpha_params = EMechanismFDMParams(
    alpha=alpha_range,
    K0=jnp.full_like(alpha_range, base_params.K0),
    E0=jnp.full_like(alpha_range, base_params.E0),
    dB=jnp.full_like(alpha_range, base_params.dB),
)

currents = vmap(fdm_solver.solve)(alpha_params)

for val, current in zip(alpha_range, currents):
    ax[0, 0].plot(fdm_solver.applied_potentials, current, label=val)
ax[0, 0].xaxis.set_inverted(True)
ax[0, 0].yaxis.set_inverted(True)
ax[0, 0].set_title(r"$\alpha$")
ax[0, 0].legend()

# K0 Varying
K0_range = jnp.array([1.0, 5.0, 10.0, 20.0, 40.0, 50.0])
K0_params = EMechanismFDMParams(
    alpha=jnp.full_like(K0_range, base_params.alpha),
    E0=jnp.full_like(K0_range, base_params.E0),
    dB=jnp.full_like(K0_range, base_params.dB),
    K0=K0_range,
)

currents = vmap(fdm_solver.solve)(K0_params)

for val, current in zip(K0_range, currents):
    ax[0, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.0f}")
ax[0, 1].xaxis.set_inverted(True)
ax[0, 1].yaxis.set_inverted(True)
ax[0, 1].set_title(r"$K_0$")
ax[0, 1].legend()

# E0 Varying
E0_range = jnp.linspace(-2.0, 2.0, 5)
E0_params = EMechanismFDMParams(
    alpha=jnp.full_like(E0_range, base_params.alpha),
    E0=E0_range,
    dB=jnp.full_like(E0_range, base_params.dB),
    K0=jnp.full_like(E0_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(E0_params)

for val, current in zip(E0_range, currents):
    ax[1, 0].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 0].xaxis.set_inverted(True)
ax[1, 0].yaxis.set_inverted(True)
ax[1, 0].set_title(r"$E_0$")
ax[1, 0].legend()

# dB Varying
dB_range = jnp.array([0.1, 0.5, 1.0, 2.0, 5.0])

dB_params = EMechanismFDMParams(
    alpha=jnp.full_like(dB_range, base_params.alpha),
    E0=jnp.full_like(dB_range, base_params.E0),
    dB=dB_range,
    K0=jnp.full_like(dB_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(dB_params)

for val, current in zip(dB_range, currents):
    ax[1, 1].plot(fdm_solver.applied_potentials, current, label=val)

ax[1, 1].xaxis.set_inverted(True)
ax[1, 1].yaxis.set_inverted(True)
ax[1, 1].set_title(r"$d_B$")
ax[1, 1].legend()

plt.show()


In [ ]:
voltammetry = CyclicDC()

fdm_solver = EMechanismFDMSolver(voltammetry)

dB_range = jnp.array([0.1, 0.5, 1.0, 2.0, 5.0])

dB_params = EMechanismFDMParams(
    alpha=jnp.full_like(dB_range, base_params.alpha),
    E0=jnp.full_like(dB_range, base_params.E0),
    dB=dB_range,
    K0=jnp.full_like(dB_range, base_params.K0),
)

currents = vmap(fdm_solver.solve)(dB_params)

for val, current in zip(dB_range, currents):
    plt.plot(fdm_solver.applied_potentials, current, label=val)

plt.title(r"$d_B$")
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.legend()
plt.show()

# Experimental Samples


In [ ]:
dc_voltammetry = LinearSweepDC()
ac_voltammetry = LinearSweepAC()
cyclic_voltammetry = CyclicDC()

# --- Sampling ---
key = jr.key(42)
generate_key, sampling_key, key = jr.split(key, 3)

dc_fdm_solver = EMechanismFDMSolver(dc_voltammetry)
ac_fdm_solver = EMechanismFDMSolver(ac_voltammetry)
cyclic_fdm_solver = EMechanismFDMSolver(cyclic_voltammetry)

true_params = EMechanismFDMParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    E0=jnp.array(2.0),
    dB=jnp.array(1.2),
)


base_dc_current = dc_fdm_solver.solve(true_params)
base_ac_current = ac_fdm_solver.solve(true_params)
base_cyclic_current = cyclic_fdm_solver.solve(true_params)

dc_samples = generate_noisy_samples(
    10,
    base_dc_current,
    0.1,
    key=key,
)

ac_samples = generate_noisy_samples(
    10,
    base_ac_current,
    0.1,
    key=key,
)

cyclic_samples = generate_noisy_samples(
    10,
    base_cyclic_current,
    0.1,
    key=key,
)


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

for sample in dc_samples:
    ax1.plot(dc_fdm_solver.applied_potentials, sample)


for sample in ac_samples:
    ax2.plot(dc_fdm_solver.applied_potentials, sample)

for sample in cyclic_samples:
    ax3.plot(cyclic_fdm_solver.applied_potentials, sample)

plt.tight_layout()
plt.show()

# Sampling


In [ ]:
def plot_e_datasets(datasets, heading: str):
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(ncols=4, figsize=(20, 5))

    true_params = EMechanismFDMParams(
        alpha=jnp.array(0.6),
        K0=jnp.array(10.0),
        E0=jnp.array(2.0),
        dB=jnp.array(1.2),
    )

    hist_kwargs = dict(
        alpha=0.75,
        bins=50,
        density=True,
    )

    for d in datasets.keys():
        ax1.hist(datasets[d]["alpha"], label=d, **hist_kwargs)
        ax2.hist(datasets[d]["K0"], **hist_kwargs)
        ax3.hist(datasets[d]["E0"], **hist_kwargs)
        ax4.hist(datasets[d]["dB"], **hist_kwargs)

    ax1.set_title("alpha")
    ax1.axvline(true_params.alpha, c="black", linestyle="--")
    ax2.set_title("K0")
    ax2.axvline(true_params.K0, c="black", linestyle="--")
    ax3.set_title("E0")
    ax3.axvline(true_params.E0, c="black", linestyle="--")
    ax4.set_title("dB")
    ax4.axvline(true_params.dB, c="black", linestyle="--")

    handles, labels = ax1.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
    )

    fig.suptitle(
        heading,
        fontsize=18,
        y=0.98,
    )

    plt.show()

    print(f"{'Method':<8} {'Param':<10} {'μ ± σ':<30}")
    print("-" * 52)

    for method, params in datasets.items():
        for param, values in params.items():
            mu = np.mean(values)
            sigma = np.std(values)
            summary = f"{mu:.4e} ± {sigma:.4e}"
            print(f"{method:<8} {param:<10} {summary:<30}")
        print("-" * 28)
    print()


## Random Walk Metropolis-Hastings


In [ ]:
dc = np.load("./data/E_MetropolisHastings_LinearSweepDC.npz")
ac = np.load("./data/E_MetropolisHastings_LinearSweepAC.npz")
cyclic = np.load("./data/E_MetropolisHastings_CyclicDC.npz")

datasets = {
    "DC": {
        "alpha": dc["alpha"].flatten(),
        "K0": dc["K0"].flatten(),
        "E0": dc["E0"].flatten(),
        "dB": dc["dB"].flatten(),
    },
    "AC": {
        "alpha": ac["alpha"].flatten(),
        "K0": ac["K0"].flatten(),
        "E0": ac["E0"].flatten(),
        "dB": ac["dB"].flatten(),
    },
    "Cyclic": {
        "alpha": cyclic["alpha"].flatten(),
        "K0": cyclic["K0"].flatten(),
        "E0": cyclic["E0"].flatten(),
        "dB": cyclic["dB"].flatten(),
    },
}

plot_e_datasets(
    datasets, heading="Posterior Distribution using Random Walk Metropolis-Hastings"
)

In [ ]:
mchmc = np.load("./data/E_MCHMC_CyclicDC.npz")
rw = np.load("./data/E_MetropolisHastings_CyclicDC.npz")

datasets = {
    "MCHMC": {
        "alpha": mchmc["alpha"].flatten(),
        "K0": mchmc["K0"].flatten(),
        "E0": mchmc["E0"].flatten(),
        "dB": mchmc["dB"].flatten(),
    },
    "RW": {
        "alpha": rw["alpha"].flatten(),
        "K0": rw["K0"].flatten(),
        "E0": rw["E0"].flatten(),
        "dB": rw["dB"].flatten(),
    },
}

plot_e_datasets(datasets, heading="Posterior Distribution using MCHMC")

In [ ]:
print("Alpha")
print(blackjax.diagnostics.potential_scale_reduction(mchmc["alpha"]))
print(blackjax.diagnostics.potential_scale_reduction(rw["alpha"]))
print("-----------")
print("K0")
print(blackjax.diagnostics.potential_scale_reduction(mchmc["K0"]))
print(blackjax.diagnostics.potential_scale_reduction(rw["K0"]))
print("-----------")
print("E0")
print(blackjax.diagnostics.potential_scale_reduction(mchmc["E0"]))
print(blackjax.diagnostics.potential_scale_reduction(rw["E0"]))
print("-----------")
print("dB")
print(blackjax.diagnostics.potential_scale_reduction(mchmc["dB"]))
print(blackjax.diagnostics.potential_scale_reduction(rw["dB"]))
print("-----------")